
# LAB | Whisper STT Implementation
## Name: Martha Olukayode
## Project Description: 
Welcome to the Whisper STT (Speech-to-Text) Implementation lab! In this exercise, you'll learn how to use OpenAI's Whisper API to transcribe audio files. Whisper is a powerful speech recognition model that can handle multiple languages, accents, and audio qualities.

In [ ]:
pip install openai pydub python-dotenv

In [1]:
# ========================================
# Step 1: Setting Up the Environment
# ========================================

import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydub import AudioSegment

# Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY was not found in .env")

# Initialize OpenAI client
client = OpenAI(api_key=api_key)

# Create required directories
audio_dir = Path("audio")
chunks_dir = Path("chunks")
output_dir = Path("output")

audio_dir.mkdir(exist_ok=True)
chunks_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

print("✓ OpenAI client initialized")
print("✓ Audio directory ready")
print("✓ Chunks directory ready")
print("✓ Output directory ready")
print("\n✓ Step 1 setup complete!")

✓ OpenAI client initialized
✓ Audio directory ready
✓ Chunks directory ready
✓ Output directory ready

✓ Step 1 setup complete!


In [2]:
# ========================================
# Step 2: Downloading Sample Meeting Audio
# ========================================

from pathlib import Path
from pydub import AudioSegment

audio_path = Path("audio/CaliforniaMoonlanding.mp3")

# Check that the file exists
if not audio_path.exists():
    raise FileNotFoundError(
        f"Audio file not found: {audio_path}"
    )

# Load the audio
audio = AudioSegment.from_file(audio_path)

# Get basic audio information
duration_seconds = len(audio) / 1000
duration_minutes = duration_seconds / 60

print("✓ Audio file loaded successfully!")
print(f"File: {audio_path.name}")
print(f"Duration: {duration_seconds:.2f} seconds")
print(f"Duration: {duration_minutes:.2f} minutes")
print(f"Channels: {audio.channels}")
print(f"Frame rate: {audio.frame_rate} Hz")
print(f"File size: {audio_path.stat().st_size / (1024 * 1024):.2f} MB")

✓ Audio file loaded successfully!
File: CaliforniaMoonlanding.mp3
Duration: 86.20 seconds
Duration: 1.44 minutes
Channels: 1
Frame rate: 44100 Hz
File size: 0.89 MB


In [3]:
# ========================================
# Step 3: Basic Transcription (Without Chunking)
# ========================================

try:
    print("Transcribing audio...")

    with open(audio_path, "rb") as audio_file:

        transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file
        )

    print("\n✓ Transcription completed successfully!")

    print("\n=== TRANSCRIPTION ===")
    print(transcription.text)

except Exception as e:
    print(f"✗ Transcription failed: {e}")

Transcribing audio...

✓ Transcription completed successfully!

=== TRANSCRIPTION ===
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think o

In [ ]:
# ========================================
# Step 4: Transcription with Prompts (Guided Approach)
# ========================================

context_prompt = """
This audio discusses California and the Moon landing.
Possible terminology and proper names may include:
NASA, Apollo, astronauts, spacecraft, Moon, California,
Neil Armstrong, Buzz Aldrin, and space exploration.
"""

try:
    print("Transcribing audio with context prompt...")

    with open(audio_path, "rb") as audio_file:

        prompted_transcription = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            prompt=context_prompt
        )

    print("\n✓ Prompted transcription completed successfully!")

    print("\n=== PROMPTED TRANSCRIPTION ===")
    print(prompted_transcription.text)

except Exception as e:
    print(f"✗ Prompted transcription failed: {e}")

Transcribing audio with context prompt...

✓ Prompted transcription completed successfully!

=== PROMPTED TRANSCRIPTION ===
It was rather interesting just to watch them gathering their materials and bouncing around, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Yes, I'll be so glad when they land back now, but I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the Moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will do some more exploring up there, because that was such a very small area, when y

In [5]:
# ========================================
# Step 5: Transcription Without Prompts (Unguided Approach)
# ========================================

unprompted_text = transcription.text
prompted_text = prompted_transcription.text

print("=== WITHOUT PROMPT ===")
print(unprompted_text)

print("\n" + "=" * 60)

print("\n=== WITH PROMPT ===")
print(prompted_text)

print("\n" + "=" * 60)

# Simple comparison
if unprompted_text == prompted_text:
    print("\nThe transcriptions are identical.")
else:
    print("\n✓ Differences were found between the transcriptions.")

=== WITHOUT PROMPT ===
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Maine. Oh, yes. I'll be so glad when they land back now. But I think that's pretty well a fact, because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was. Have they met with the one that was circling? Yes, they've rendezvoused. So I understand. That wasn't shown either, but they say they have rendezvoused. So that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them sending up ships? I think they will. I think they will do some more exploring up there. Very positive. Because that was such a very small area, when you think of it, that they just gathered rocks and samples of soil and all

In [6]:
comparison_file = output_dir / "transcription_comparison.txt"

with open(comparison_file, "w", encoding="utf-8") as file:

    file.write("WITHOUT PROMPT\n")
    file.write("=" * 50 + "\n")
    file.write(unprompted_text)

    file.write("\n\nWITH PROMPT\n")
    file.write("=" * 50 + "\n")
    file.write(prompted_text)

print(f"✓ Comparison saved to: {comparison_file}")

✓ Comparison saved to: output\transcription_comparison.txt


In [7]:
# ========================================
# Step 6: Implementing Audio Chunking
# ========================================

def split_audio_into_chunks(
    audio_path,
    chunks_dir,
    chunk_length_minutes=0.5
):
    """
    Split an audio file into smaller chunks.

    Args:
        audio_path: Path to the original audio file.
        chunks_dir: Directory where chunks will be saved.
        chunk_length_minutes: Length of each chunk in minutes.

    Returns:
        List of paths to the generated audio chunks.
    """

    audio = AudioSegment.from_file(audio_path)

    # pydub measures time in milliseconds
    chunk_length_ms = int(
        chunk_length_minutes * 60 * 1000
    )

    chunk_files = []

    for i, start_ms in enumerate(
        range(0, len(audio), chunk_length_ms)
    ):

        end_ms = min(
            start_ms + chunk_length_ms,
            len(audio)
        )

        chunk = audio[start_ms:end_ms]

        chunk_path = chunks_dir / f"chunk_{i + 1}.mp3"

        chunk.export(
            chunk_path,
            format="mp3"
        )

        chunk_files.append(chunk_path)

        print(
            f"✓ Chunk {i + 1}: "
            f"{start_ms / 1000:.1f}s - "
            f"{end_ms / 1000:.1f}s"
        )

    return chunk_files

In [8]:
chunk_files = split_audio_into_chunks(
    audio_path,
    chunks_dir,
    chunk_length_minutes=0.5
)

print("\n=================================")
print("CHUNKING COMPLETE")
print("=================================")
print(f"Total chunks created: {len(chunk_files)}")

✓ Chunk 1: 0.0s - 30.0s
✓ Chunk 2: 30.0s - 60.0s
✓ Chunk 3: 60.0s - 86.2s

CHUNKING COMPLETE
Total chunks created: 3


In [9]:
# ========================================
# Step 7: Transcribe Chunks with Timestamps
# ========================================

timestamped_segments = []

chunk_length_seconds = 30

for i, chunk_path in enumerate(chunk_files):

    print(f"\nTranscribing {chunk_path.name}...")

    try:
        with open(chunk_path, "rb") as audio_file:

            response = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file,
                response_format="verbose_json",
                timestamp_granularities=["segment"]
            )

        # Calculate where this chunk starts
        # in the original recording
        chunk_offset = i * chunk_length_seconds

        for segment in response.segments:

            adjusted_start = segment.start + chunk_offset
            adjusted_end = segment.end + chunk_offset

            timestamped_segments.append({
                "start": adjusted_start,
                "end": adjusted_end,
                "text": segment.text.strip()
            })

        print(f"✓ {chunk_path.name} completed")

    except Exception as e:
        print(f"✗ Error transcribing {chunk_path.name}: {e}")


Transcribing chunk_1.mp3...
✓ chunk_1.mp3 completed

Transcribing chunk_2.mp3...
✓ chunk_2.mp3 completed

Transcribing chunk_3.mp3...
✓ chunk_3.mp3 completed


In [10]:
print("\n=== COMPLETE TIMESTAMPED TRANSCRIPTION ===\n")

for segment in timestamped_segments:

    print(
        f"[{segment['start']:.2f}s - "
        f"{segment['end']:.2f}s] "
        f"{segment['text']}"
    )

print(
    f"\n✓ Total timestamped segments: "
    f"{len(timestamped_segments)}"
)


=== COMPLETE TIMESTAMPED TRANSCRIPTION ===

[0.00s - 7.00s] It was rather interesting just to watch them gathering their materials and bouncing around
[7.00s - 12.50s] their, what do they call it, kangaroo walk or something like that.
[12.50s - 13.50s] Who named it that?
[13.50s - 14.50s] I don't know.
[14.50s - 19.00s] I bet those men are going to get quite a reception when they get back to New York.
[19.00s - 28.00s] Oh yes, I'll be so glad when they land back now, but I think that's pretty well fact because
[28.00s - 30.00s] they've landed so many safely now.
[30.00s - 35.00s] and I feel relieved. Just getting off of the moon was the thing that was...
[35.00s - 38.00s] Have they met with the one that was circling?
[38.00s - 44.00s] Yes, they've rendezvoused. So I understand. That wasn't shown either, so I...
[44.00s - 47.00s] But they say they have rendezvoused, so...
[47.00s - 53.00s] That's a matter of making the circles and then coming down.
[53.00s - 57.00s] What do you sort of

In [11]:
# ========================================
# Step 8: Exporting with Timestamps
# ========================================

def format_timestamp(seconds):
    """Convert seconds to HH:MM:SS format."""

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


def format_srt_timestamp(seconds):
    """Convert seconds to SRT HH:MM:SS,mmm format."""

    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    milliseconds = int((seconds - int(seconds)) * 1000)

    return (
        f"{hours:02d}:{minutes:02d}:"
        f"{secs:02d},{milliseconds:03d}"
    )

In [12]:
txt_path = output_dir / "transcription_with_timestamps.txt"

with open(txt_path, "w", encoding="utf-8") as file:

    file.write("California Moon Landing - Transcription\n")
    file.write("=" * 50 + "\n\n")

    for segment in timestamped_segments:

        start = format_timestamp(segment["start"])
        end = format_timestamp(segment["end"])

        file.write(
            f"[{start} - {end}] "
            f"{segment['text']}\n"
        )

print(f"✓ TXT exported: {txt_path}")

✓ TXT exported: output\transcription_with_timestamps.txt


In [13]:
srt_path = output_dir / "transcription.srt"

with open(srt_path, "w", encoding="utf-8") as file:

    for index, segment in enumerate(
        timestamped_segments,
        start=1
    ):

        start = format_srt_timestamp(segment["start"])
        end = format_srt_timestamp(segment["end"])

        file.write(f"{index}\n")
        file.write(f"{start} --> {end}\n")
        file.write(f"{segment['text']}\n\n")

print(f"✓ SRT exported: {srt_path}")

✓ SRT exported: output\transcription.srt


In [14]:
import json

json_path = output_dir / "transcription.json"

with open(json_path, "w", encoding="utf-8") as file:

    json.dump(
        timestamped_segments,
        file,
        indent=4,
        ensure_ascii=False
    )

print(f"✓ JSON exported: {json_path}")

✓ JSON exported: output\transcription.json


In [15]:
print("\n=================================")
print("EXPORT COMPLETE")
print("=================================")

print(f"TXT exists:  {txt_path.exists()}")
print(f"SRT exists:  {srt_path.exists()}")
print(f"JSON exists: {json_path.exists()}")

print("\nFiles created:")
print(f"✓ {txt_path}")
print(f"✓ {srt_path}")
print(f"✓ {json_path}")


EXPORT COMPLETE
TXT exists:  True
SRT exists:  True
JSON exists: True

Files created:
✓ output\transcription_with_timestamps.txt
✓ output\transcription.srt
✓ output\transcription.json
